# 01-9_remove_wcs_in_HDUL_ys

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module version_information is installed


### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [2]:
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

This notebook was generated at 2023-01-29 19:45:15 (대한민국 표준시 = GMT+0900) 
0 Python     3.9.7 64bit [MSC v.1916 64 bit (AMD64)]
1 IPython    7.31.1
2 OS         Windows 10 10.0.22621 SP0
3 numpy      1.21.5
4 pandas     1.4.1
5 matplotlib 3.5.1
6 scipy      1.7.3
7 astropy    5.0
8 photutils  1.5.0
9 ccdproc    2.3.1
10 version_information 1.0.4


### import modules

In [3]:
import os
from glob import glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import astropy.units as u
from astropy.stats import sigma_clip
from astropy.io import fits
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

from snuo1mpy import Preprocessor

import astro_utilities
import Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

ImportError: cannot import name 'PositiveScalarAngle' from 'photutils.aperture.attributes' (c:\Users\Kiehyun\anaconda3\envs\astro_Python_win_env\lib\site-packages\photutils\aperture\attributes.py)

In [ ]:
#%%
BASEDIR = astro_utilities.base_dir

BASEDIRs = sorted(Python_utilities.getFullnameListOfsubDir(BASEDIR))
print ("BASEDIRs: {}".format(BASEDIRs))
print ("len(BASEDIRs): {}".format(len(BASEDIRs)))


BASEDIR = Path(BASEDIRs[1])
print ("Starting...\n{}".format(BASEDIR))

BASEDIR = Path(BASEDIR)

MASTERDIR = BASEDIR / astro_utilities.master_dir

if not MASTERDIR.exists():
    os.makedirs("{}".format(str(MASTERDIR)))
    print("{} is created...".format(str(MASTERDIR)))

BASEDIRs: ['R:\\CCD_obs\\RiLA600_2022\\BARNARD174_Light_-_2022-10-18_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\ESO535-5_Light_-_2022-10-18_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\ESO606-10_Light_-_2022-10-18_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\IC405_Light_-_2022-10-07_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\IC410_Light_-_2022-10-07_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\M103_Light_-_2022-10-12_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\M13_Light_-_2022-10-12_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\M15_Light_-_2022-10-13_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\M27_Light_-_2022-10-12_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\M27_Light_-_2022-10-13_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\M2_Light_-_2022-10-13_-_RiLA600_STX-16803_-_1bin/', 'R:\\CCD_obs\\RiLA600_2022\\M34_Light_-_2022-10-18_-_RiLA600_STX-16803_-_1bi

In [ ]:

summary = yfu.make_summary(BASEDIR/"*.fit*")
#print(summary)
print("len(summary):", len(summary))
print("summary:", summary)
#print(summary["file"][0])

All 66 keywords (guessed from R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-10-18_-_RiLA600_STX-16803_-_1bin\ESO535-5_Light_v_2022-10-18-15-08-10_030sec_RiLA600_STX-16803_-30C_1bin.fit) will be loaded.
len(summary): 10
summary:                                                 file  filesize  SIMPLE  \
0  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
1  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
2  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
3  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
4  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
5  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
6  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
7  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
8  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
9  R:\CCD_obs\RiLA600_2022\ESO53

In [ ]:
df_light = summary[summary["IMAGETYP"] == "LIGHT"]
print ("df_light: {}".format(df_light))
print ("len(df_light): {}".format(len(df_light)))


df_light:                                                 file  filesize  SIMPLE  \
0  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
1  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
2  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
3  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
4  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
5  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
6  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
7  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
8  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   
9  R:\CCD_obs\RiLA600_2022\ESO535-5_Light_-_2022-...  33560640    True   

   BITPIX  NAXIS  NAXIS1  NAXIS2 IMAGETYP  EXPOSURE  EXPTIME  ...    CTYPE2  \
0      16      2    4096    4096    LIGHT      30.0     30.0  ...  DEC--TAN   
1      16      2 

In [4]:
for _, row in df_light.iterrows():
    # 파일명 출력
    print (row["file"])
    # fits hedaer 에 있는 wcs 정보를 지운다
    yfu.wcsremove(row["file"], 
                #additional_keys=["COMMENT"],
                output = row["file"], 
                overwrite = True)

NameError: name 'df_light' is not defined